In [1]:
import os
import subprocess

import numpy
import pandas as pd
from osgeo import gdal, gdalconst

import seabeepy as sb
from seabeepy.config import SETTINGS

In [2]:
minio_client = sb.storage.minio_login(
    user=SETTINGS.MINIO_ACCESS_ID, password=SETTINGS.MINIO_SECRET_KEY
)

In [3]:
temp_dir = r"/home/notebook/cogs"

In [4]:
version = "1-2"

# Levels to publish
levels = [1]

mission_list = [
    r"/home/notebook/shared-seabee-ns9879k/nrdata/marint_naturkart/arendal_hovekilen_2025",
]

In [5]:
# Gets codes and names for annotation 'version'
df = sb.anno.get_class_codes(version)

# Publish selected levels
for level in levels:
    # Filter codes to level of interest
    code_len = level * 2
    df = df[df["code"].str.len() == code_len].drop_duplicates().reset_index()

    # Process mission data
    for mission_dir in mission_list:
        mission_name = sb.ortho.get_layer_name(mission_dir)
        for res_type in ["classifications", "pvalues"]:
            layer_name = f"{mission_name}_{res_type}_level{level}"
            print(f"\n################\nProcessing: {layer_name}")

            print("Standardising raster.")
            if res_type == "classifications":
                res_path = os.path.join(
                    mission_dir, "results", f"image_1_lev{level}.tif"
                )
            else:
                res_path = os.path.join(
                    mission_dir, "results", f"image_1_pvalue_lev{level}.tif"
                )
            temp_path = os.path.join(temp_dir, f"{layer_name}.tif")
            stan_path = os.path.join(mission_dir, "results", f"{layer_name}.tif")
            nodata = sb.geo.get_geotiff_info(res_path)["nodata_value"]
            cmd = [
                "gdal_translate",
                "-of",
                "COG",
                "-co",
                "COMPRESS=LZW",
                "-co",
                "PREDICTOR=2",
                "-co",
                "NUM_THREADS=4",
                "-co",
                "OVERVIEWS=IGNORE_EXISTING",
                "-co",
                "BIGTIFF=YES",
                "-a_nodata",
                str(nodata),
                res_path,
                temp_path,
            ]
            subprocess.check_call(cmd)

            # Copy to MinIO and delete temporary file
            sb.storage.copy_file(temp_path, stan_path, minio_client, overwrite=True)
            os.remove(temp_path)

            print("Uploading to GeoServer.")
            if res_type == "classifications":
                sld_name = f"results_classes_v{version}_level{level}"
            else:
                sld_name = "pvalues"
            sb.geo.upload_raster_to_geoserver(
                stan_path,
                SETTINGS.GEOSERVER_USER,
                SETTINGS.GEOSERVER_PASSWORD,
                workspace="geonode",
                sld_name=sld_name,
            )

            print("Publishing to GeoNode.")
            sb.geo.publish_to_geonode(
                layer_name,
                SETTINGS.GEONODE_USER,
                SETTINGS.GEONODE_PASSWORD,
                workspace="geonode",
            )

            print("Updating metadata.")
            date = sb.ortho.parse_mission_data(mission_dir, parse_date=True)[2]
            abstract = f"Preliminary habitat {res_type} for '{mission_name}'."
            metadata = {
                "abstract": abstract,
                "date": date.isoformat(),
                "date_type": "creation",
                "attribution": "SeaBee",
            }
            sb.geo.update_geonode_metadata(
                layer_name,
                SETTINGS.GEONODE_USER,
                SETTINGS.GEONODE_PASSWORD,
                metadata,
            )


################
Processing: niva-hovekilen_hovekilen_202506050831_rgb_120m_classifications_level1
Standardising raster.
Input file size is 18150, 20951
0...10...20...30...40...50...60...70...80...90...100 - done.
Uploading to GeoServer.
If you want to use a different style, either delete/update the existing version, or create an SLD file with a different name.
Publishing to GeoNode.
Updating metadata.

################
Processing: niva-hovekilen_hovekilen_202506050831_rgb_120m_pvalues_level1
Standardising raster.
Input file size is 18150, 20951
0...10...20...30...40...50...60...70...80...90...100 - done.
Uploading to GeoServer.
Publishing to GeoNode.
Updating metadata.
